# Naive Bayes
## Real-world scenario: Spam email detection

An email provider wants to label a message as **spam (1)** or **ham / not spam (0)** based on its text. Naive Bayes is fast and famously effective for text classification, so it is the classic choice for spam filters.

### Step 1 - Import the libraries

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

np.random.seed(42)

### Step 2 - Create a small, realistic dataset
A handful of short emails labelled spam or ham. We add a duplicate and a blank message to clean.

In [ ]:
emails = [
    ('Win a FREE iPhone now, click this link!!!', 1),
    ('Congratulations you won a lottery prize claim now', 1),
    ('Cheap meds and free money offer just for you', 1),
    ('Limited time offer, buy now and get 90% discount', 1),
    ('Claim your free gift card today urgent', 1),
    ('Hi team, please find attached the report', 0),
    ('Are we still meeting for lunch tomorrow?', 0),
    ('The project deadline is next Friday', 0),
    ('Thanks for your help on the presentation', 0),
    ('Can you review my code before I merge it?', 0),
    ('Reminder: dentist appointment at 3pm', 0),
    ('Free free free money click win prize now', 1),
]
df = pd.DataFrame(emails, columns=['text', 'label'])

# Messy data on purpose: a duplicate and an empty message
df = pd.concat([df, df.iloc[[0]]], ignore_index=True)
df.loc[len(df)] = ['', 0]
df.head()

### Step 3 - Explore the data

In [ ]:
print('Shape:', df.shape)
print('\nDuplicates:', df.duplicated().sum())
print('\nEmpty messages:', (df['text'].str.strip() == '').sum())
print('\nSpam vs ham:\n', df['label'].value_counts())

### Step 4 - Clean the data
Remove duplicates and empty messages, and lowercase the text so 'Free' and 'free' match.

In [ ]:
df = df.drop_duplicates().reset_index(drop=True)
df = df[df['text'].str.strip() != ''].reset_index(drop=True)
df['text'] = df['text'].str.lower()
print('Rows after cleaning:', len(df))

### Step 5 - Turn text into numbers
Models cannot read words, so `CountVectorizer` converts each email into word counts (a 'bag of words').

In [ ]:
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(df['text'])   # feature matrix of word counts
y = df['label']
print('Vocabulary size:', len(vectorizer.get_feature_names_out()))

### Step 6 - Train / test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42)

### Step 7 - Train the Naive Bayes model

In [ ]:
model = MultinomialNB()
model.fit(X_train, y_train)

### Step 8 - Evaluate

In [ ]:
y_pred = model.predict(X_test)
print('Accuracy:', round(accuracy_score(y_test, y_pred), 3))
print('\nConfusion matrix:\n', confusion_matrix(y_test, y_pred))
print('\nReport:\n', classification_report(y_test, y_pred, zero_division=0))

### Step 9 - Classify brand-new emails

In [ ]:
new_emails = ['win a free prize now click here',
              'can we reschedule the meeting to monday']
new_X = vectorizer.transform([e.lower() for e in new_emails])
for email, pred in zip(new_emails, model.predict(new_X)):
    print(('SPAM' if pred == 1 else 'HAM '), '->', email)